# Extreme OOD Probe Suite — LLaVA-RLHF

Run **LLaVA-RLHF** (7B, RLHF LoRA) on 7 extreme out-of-distribution probes.

**Environment:** `conda activate llava_rlhf`

**Before running inference:** generate synthetic images with `python generate_images.py` .

**Output:** `llava_rlhf_extreme_results.json`


## Prepare images

```bash
cd mywork/extreme_case
python generate_images.py
```

Writes `pure_noise.jpg`, `solid_color.jpg`, `geometric_circle.jpg`, and `test_noisy.jpg` to `mywork/test_images/`.

In [1]:
import json
import os
from pathlib import Path

import torch
from PIL import Image
from peft import PeftModel
from llava.constants import DEFAULT_IMAGE_TOKEN, IMAGE_TOKEN_INDEX
from llava.mm_utils import process_images, tokenizer_image_token
from llava.model.builder import load_pretrained_model

# ==================== Configuration ====================
# NOTEBOOK_DIR: run from mywork/extreme_case/
# IMAGE_DIR:    synthetic probe images (generate_images.py → mywork/test_images/)
# SFT_PATH:     LLaVA-RLHF SFT+ backbone (8-bit load)
# LORA_PATH:    RLHF LoRA adapter applied on top of SFT+
# OUTPUT_JSON:  consumed by compare_extreme_results.py

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent
IMAGE_DIR = NOTEBOOK_DIR.parent / "test_images"
SFT_PATH = (PROJECT_ROOT / "LLaVA-RLHF/models/LLaVA-RLHF-7B-v1.5-224/sft_model").resolve()
LORA_PATH = (PROJECT_ROOT / "LLaVA-RLHF/models/LLaVA-RLHF-7B-v1.5-224/rlhf_lora_adapter_model").resolve()
OUTPUT_JSON = NOTEBOOK_DIR / "llava_rlhf_extreme_results.json"

print(f"Notebook dir: {NOTEBOOK_DIR}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Image dir:    {IMAGE_DIR}")
print(f"SFT path:     {SFT_PATH}")
print(f"LoRA path:    {LORA_PATH}")


/data/workspaces/zwang829/conda_envs/llava_rlhf/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setting ds_accelerator to cuda (auto detect)
Notebook dir: /data/workspaces/zwang829/workspace/algorithm/LLaVA_RLHF/mywork/extreme_case
Project root: /data/workspaces/zwang829/workspace/algorithm/LLaVA_RLHF
Image dir:    /data/workspaces/zwang829/workspace/algorithm/LLaVA_RLHF/mywork/test_images
SFT path:     /data/workspaces/zwang829/workspace/algorithm/LLaVA_RLHF/LLaVA-RLHF/models/LLaVA-RLHF-7B-v1.5-224/sft_model
LoRA path:    /data/workspaces/zwang829/workspace/algorithm/LLaVA_RLHF/LLaVA-RLHF/models/LLaVA-RLHF-7B-v1.5-224/rlhf_lora_adapter_model


In [2]:
# ==================== Probe definitions & scoring (schema v1.0) ====================
# Categories: A=noise, B=solid color, C=perturbed real image, D=geometry
# A2 is qualitative (manual review only); scored accuracy excludes A2 (6 probes).
QUALITATIVE_IDS = {"A2"}

PROBES = [
    {
        "image": "pure_noise.jpg",
        "id": "A1",
        "type": "existence",
        "question": "Is this a photograph of a real scene? Answer yes or no.",
        "gt": "no",
        "note": "Pure noise has no real scene; checks scene fabrication on zero semantics",
    },
    {
        "image": "pure_noise.jpg",
        "id": "A2",
        "type": "qualitative",
        "question": "Describe this image in detail.",
        "gt": "noise",
        "note": "Qualitative only: inspect whether noise is narrativized into objects (not auto-scored)",
    },
    {
        "image": "solid_color.jpg",
        "id": "B1",
        "type": "attribute",
        "question": "What color is this? Answer with one word.",
        "gt": "blue",
        "note": "Flat steelblue field; basic color sanity check",
    },
    {
        "image": "solid_color.jpg",
        "id": "B2",
        "type": "existence",
        "question": "Is there a sky in this image? Answer yes or no.",
        "gt": "no",
        "note": "Solid color must not be projected into a sky scene",
    },
    {
        "image": "test_noisy.jpg",
        "id": "C1",
        "type": "count",
        "question": "How many tissue boxes are in this image? Answer with a number.",
        "gt": "1",
        "note": "Light noise on test.jpg; count should remain 1",
    },
    {
        "image": "test_noisy.jpg",
        "id": "C2",
        "type": "existence",
        "question": "Is there a keyboard in this image? Answer yes or no.",
        "gt": "no",
        "note": "Noisy office scene; keyboard context trap should stay negative",
    },
    {
        "image": "geometric_circle.jpg",
        "id": "D1",
        "type": "existence",
        "question": "Is this a drawing of an animal? Answer yes or no.",
        "gt": "no",
        "note": "Simple circle; checks sun/animal shape priors",
    },
]

SCHEMA_VERSION = "1.0"
BENCHMARK_NAME = "extreme_ood_probe"


def judge_correctness(pred, gt, qtype):
    """Score a model answer against ground truth for auto-graded probes.

    Args:
        pred: Model-generated answer text.
        gt: Ground-truth string (e.g. yes/no, color, count).
        qtype: Probe type (qualitative probes return None).

    Returns:
        True if correct, False if wrong, or None for qualitative probes.
    """
    pred_lower = pred.lower().strip(".")
    gt_lower = gt.lower().strip()
    if qtype == "qualitative":
        return None
    if gt_lower in ["yes", "no"]:
        if gt_lower == "yes":
            return "yes" in pred_lower and "no" not in pred_lower
        return "no" in pred_lower and "yes" not in pred_lower
    return gt_lower in pred_lower


def _verdict_label(correct, qtype, probe_id):
    """Map correctness to a human-readable verdict label.

    Args:
        correct: Result from judge_correctness (or None).
        qtype: Probe type string.
        probe_id: Probe id (e.g. A2 is always qualitative).

    Returns:
        One of "qualitative", "n/a", "correct", or "wrong".
    """
    if probe_id in QUALITATIVE_IDS or qtype == "qualitative":
        return "qualitative"
    if correct is None:
        return "n/a"
    return "correct" if correct else "wrong"


def build_benchmark_output(run_id, models, probes, answers_by_model):
    """Assemble the v1.0 benchmark JSON structure from raw answers.

    Args:
        run_id: Identifier for this run (e.g. llava_rlhf_extreme).
        models: List of model metadata dicts with "key", "name", "path".
        probes: Probe definition list (PROBES).
        answers_by_model: Dict mapping model key -> probe id -> answer text.

    Returns:
        Dict ready for JSON serialization (schema_version, summary, per_probe, ...).
    """
    model_keys = [m["key"] for m in models]
    per_probe = []
    for probe in probes:
        answers, correct, verdict = {}, {}, {}
        for key in model_keys:
            ans = answers_by_model[key].get(probe["id"], "")
            ok = judge_correctness(ans, probe["gt"], probe["type"])
            answers[key] = ans
            correct[key] = ok
            verdict[key] = _verdict_label(ok, probe["type"], probe["id"])
        per_probe.append({**probe, "answers": answers, "correct": correct, "verdict": verdict})

    by_model = {}
    for key in model_keys:
        scored = [r for r in per_probe if r["id"] not in QUALITATIVE_IDS]
        c = sum(1 for r in scored if r["correct"][key])
        t = len(scored)
        by_type = {}
        for r in per_probe:
            if r["id"] in QUALITATIVE_IDS:
                continue
            q = r["type"]
            by_type.setdefault(q, {"correct": 0, "wrong": 0})
            if r["correct"][key]:
                by_type[q]["correct"] += 1
            else:
                by_type[q]["wrong"] += 1
        by_model[key] = {
            "correct": c,
            "wrong": t - c,
            "accuracy": round(c / max(t, 1), 4),
            "qualitative_ids": sorted(QUALITATIVE_IDS),
            "by_type": by_type,
        }

    summary = {
        "total": len(per_probe),
        "scored_total": len([p for p in probes if p["id"] not in QUALITATIVE_IDS]),
        "by_model": by_model,
    }

    return {
        "schema_version": SCHEMA_VERSION,
        "benchmark": BENCHMARK_NAME,
        "run_id": run_id,
        "models": models,
        "probes": probes,
        "per_probe": per_probe,
        "summary": summary,
    }


def save_benchmark_output(output, path):
    """Write benchmark dict to a UTF-8 JSON file.

    Args:
        output: Dict from build_benchmark_output.
        path: Filesystem path for the output .json file.

    Returns:
        None.
    """
    with open(path, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2, ensure_ascii=False)


In [3]:
def generate_answer(model, tokenizer, image_processor, image_path, question):
    """Run greedy LLaVA generation for one (image, question) pair.

    Args:
        model: Loaded LLaVA model (SFT+ with optional LoRA).
        tokenizer: LLaVA tokenizer.
        image_processor: Vision preprocessor from load_pretrained_model.
        image_path: Path to probe image file.
        question: User question text (without image token).

    Returns:
        Decoded assistant reply string (special tokens stripped).
    """
    image = Image.open(image_path).convert("RGB")
    images_tensor = process_images([image], image_processor, model.config)
    images_tensor = images_tensor.to(model.device, dtype=torch.float16)

    # LLaVA v1.5 chat format: USER block with DEFAULT_IMAGE_TOKEN, then ASSISTANT:
    prompt = f"USER: {DEFAULT_IMAGE_TOKEN}\n{question}\nASSISTANT:"
    input_ids = tokenizer_image_token(
        prompt, tokenizer, IMAGE_TOKEN_INDEX, return_tensors="pt"
    ).unsqueeze(0)
    input_ids = input_ids.to(model.device)

    # Longer budget for open-ended "Describe..." probes; short answers otherwise
    max_new_tokens = 128 if question.startswith("Describe") else 32

    # Greedy decode (do_sample=False) for reproducible probe scores
    with torch.inference_mode():
        output_ids = model.generate(
            input_ids=input_ids,
            images=images_tensor,
            do_sample=False,
            temperature=0.2,
            max_new_tokens=max_new_tokens,
            use_cache=False,
        )

    return tokenizer.decode(output_ids[0, input_ids.shape[1]:], skip_special_tokens=True).strip()


In [4]:
# ==================== Load model, run probes, save JSON ====================
# Load order: SFT+ backbone → RLHF LoRA. Console prints truncate answers; JSON stores full text.

print("=" * 80)
print("Loading SFT+ base, then RLHF LoRA adapter...")
print("=" * 80)

tokenizer, model, image_processor, context_len = load_pretrained_model(
    model_path=str(SFT_PATH),
    model_base=None,
    model_name="llava-v1.5-7b",
    load_8bit=True,
    device_map="auto",
)
model = model.eval()

model = PeftModel.from_pretrained(model, str(LORA_PATH), device_map="auto")
model = model.eval()

print("\nRunning LLaVA-RLHF on extreme OOD probes...")
results_rlhf = {}
for p in PROBES:
    img_path = IMAGE_DIR / p["image"]
    if not img_path.is_file():
        print(f"[SKIP] {img_path} not found — run generate_images.py first")
        continue

    ans = generate_answer(model, tokenizer, image_processor, str(img_path), p["question"])
    results_rlhf[p["id"]] = ans

    if p["id"] in QUALITATIVE_IDS:
        print(f"[RLHF][{p['id']}] {p['image'][:18]:<18} | (qualitative) | {ans[:80]}")
        continue

    ok = judge_correctness(ans, p["gt"], p["type"])
    status = "OK" if ok else "WRONG"
    print(f"[RLHF][{p['id']}] {p['image'][:18]:<18} | {status:<5} | {ans[:60]}")

benchmark_output = build_benchmark_output(
    run_id="llava_rlhf_extreme",
    models=[
        {
            "key": "llava_rlhf",
            "name": "LLaVA-RLHF (SFT+ + RLHF LoRA)",
            "path": str(LORA_PATH),
        }
    ],
    probes=PROBES,
    answers_by_model={"llava_rlhf": results_rlhf},
)
per_probe = benchmark_output["per_probe"]
stats = benchmark_output["summary"]["by_model"]["llava_rlhf"]

print("\n" + "=" * 100)
print("LLaVA-RLHF EXTREME OOD PROBE RESULTS")
print("=" * 100)
print(f"{'ID':<4} {'Image':<18} {'Type':<12} {'GT':<10} {'Answer':<35} {'Verdict'}")
print("-" * 100)
for row in per_probe:
    ans = row["answers"]["llava_rlhf"]
    verdict = row["verdict"]["llava_rlhf"]
    print(
        f"{row['id']:<4} {row['image'][:17]:<18} {row['type']:<12} "
        f"{row['gt']:<10} {ans[:34]:<35} {verdict}"
    )
print("-" * 100)
print(
    f"\nScored accuracy (A2 excluded): {stats['correct']}/{stats['correct'] + stats['wrong']} "
    f"({stats['accuracy']:.1%})"
)
print("=" * 100)

save_benchmark_output(benchmark_output, OUTPUT_JSON)
print(f"\n[OK] Saved {OUTPUT_JSON}")


Loading SFT+ base, then RLHF LoRA adapter...


/data/workspaces/zwang829/conda_envs/llava_rlhf/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
The model weights are not tied. Please use the `tie_weights` method before using the `infer_auto_device` function.
Loading checkpoint shards: 100%|██████████| 2/2 [00:06<00:00,  3.09s/it]
Some weights of the model checkpoint at /data/workspaces/zwang829/workspace/algorithm/LLaVA_RLHF/LLaVA-RLHF/models/LLaVA-RLHF-7B-v1.5-224/sft_model were not used when initializing LlavaLlamaForCausalLM: ['model.vision_tower.vision_tower.vision_model.encoder.layers.23.mlp.fc1.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.20.mlp.fc2.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.20.self_attn.k_proj.bias', 'model.vision_tower.vision_tower.vision_


Running LLaVA-RLHF on extreme OOD probes...
[RLHF][A1] pure_noise.jpg     | OK    | No. This is not a photograph of a real scene. It is a close-
[RLHF][A2] pure_noise.jpg     | (qualitative) | The image is a close-up of a colorful, abstract pattern, which appears to be a b
[RLHF][B1] solid_color.jpg    | OK    | Blue
[RLHF][B2] solid_color.jpg    | OK    | No
[RLHF][C1] test_noisy.jpg     | WRONG | There are two tissue boxes in this image.
[RLHF][C2] test_noisy.jpg     | WRONG | Yes, there is a keyboard in this image.
[RLHF][D1] geometric_circle.j | OK    | No

LLaVA-RLHF EXTREME OOD PROBE RESULTS
ID   Image              Type         GT         Answer                              Verdict
----------------------------------------------------------------------------------------------------
A1   pure_noise.jpg     existence    no         No. This is not a photograph of a   correct
A2   pure_noise.jpg     qualitative  noise      The image is a close-up of a color  qualitative
B1   solid_co

## Next steps

1. Open **`extreme_ov.ipynb`** in the **`llava-ov`** Conda env and run all cells.
2. After both `llava_rlhf_extreme_results.json` and `llava_ov_extreme_results.json` exist, run:

```bash
python compare_extreme_results.py
```
